# Monthly Revenue Forecasting: Nominal and Inflation-Adjusted Real Revenue

This notebook builds a forecasting workflow for monthly revenue.

The dataset is a monthly revenue mart covering 2021–2023. The goal is to forecast monthly **2024 nominal revenue** and **2024 inflation-adjusted real revenue**, then calculate yearly totals.

The modeling approach is:

1. Compare a **log trend model**
2. Compare a **log trend + sine/cosine seasonality model**
3. Compare both against a simple **last-value baseline**
4. Evaluate on a chronological validation split
5. Refit the selected model on all available data
6. Forecast 2024

The project avoids over-engineering because the dataset is small: only about 30 monthly observations.

## 1. Setup

This notebook uses:

- `bigquery` module from `google.cloud` to setup BigQuery client
- `pandas` and `numpy` for data handling
- `plotly.express` for visualization
- `scikit-learn` for model fitting and evaluation
- `pandas_gbq` for ingesting the results into BigQuery

In [22]:
# !/usr/bin/env python3

# Imports
from google.cloud import bigquery
import pandas_gbq

import pandas as pd
import numpy as np

import plotly.express as px
import plotly.io as pio

from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)

# Plotly renderer setting.
# If charts do not display in your environment, try:
# pio.renderers.default = "notebook"
# or:
# pio.renderers.default = "browser"
pio.renderers.default = "notebook_connected"

pd.set_option("display.float_format", "{:,.2f}".format)

## 2. Load the monthly revenue mart

Query the monthly revenue mart via BigQuery client and save it to a DataFrame.

In [23]:
# BigQuery client setup
PROJECT_ID = "superstore-analysis-496710"
LOCATION = "EU"
DATASET = "dbt_doruk"

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

# SQL query to fetch the data
sql = f"""
select
    order_month,
    nominal_revenue,
    real_revenue,
    order_count,
    customer_count,
    units_sold,
    units_per_order,
    nominal_avg_order_value,
    real_avg_order_value,
    cpi_index_jan_2021_100,
    inflation_adjustment_factor
from `{PROJECT_ID}.{DATASET}.fct_monthly_revenue`
where order_month between date '2021-01-01' and date '2023-06-01'
order by order_month
"""

# Execute the query and load the results into a DataFrame
df = client.query(sql).to_dataframe()
df["order_month"] = pd.to_datetime(df["order_month"])
df = df.sort_values("order_month").reset_index(drop=True)

# Convert columns to numeric, coercing errors to NaN
numeric_cols = [
    "nominal_revenue",
    "real_revenue",
    "order_count",
    "customer_count",
    "units_sold",
    "avg_order_value",
    "real_avg_order_value",
    "cpi_value",
    "inflation_adjustment_factor"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

/Users/doruk/dev/3a-superstore-analysis/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [24]:
# Data overview
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print(f"Date range: {df['order_month'].min().date()} to {df['order_month'].max().date()}")

df.head()

Rows: 30
Columns: 11
Date range: 2021-01-01 to 2023-06-01


,order_month,nominal_revenue,real_revenue,order_count,customer_count,units_sold,units_per_order,nominal_avg_order_value,real_avg_order_value,cpi_index_jan_2021_100,inflation_adjustment_factor
0,2021-01-01,"262,374,739.35","262,374,739.35",332774,96537,7482261,22.48,788.447232506,788.45,100.000000000,1.00
1,2021-02-01,"245,348,178.10","243,140,821.27",300211,94975,6752817,22.49,817.252459437,809.90,100.907851200,0.99
2,2021-03-01,"280,547,713.26","275,065,691.15",332385,96373,7469408,22.47,844.044446230,827.55,101.992986600,0.98
3,2021-04-01,"281,199,642.31","271,152,269.98",321862,96019,7233028,22.47,873.665242588,842.45,103.705435400,0.96
4,2021-05-01,"300,710,698.34","287,412,348.08",332218,96317,7482198,22.52,905.160762933,865.13,104.626923800,0.96


## 3. Basic checks

This is not a full EDA section. The goal is only to confirm that the monthly time series is usable for modeling.

Important checks:

- Are there missing values?
- Are there missing months?
- Do the target columns exist?

In [25]:
required_columns = [
    "order_month",
    "nominal_revenue",
    "real_revenue"
]

missing_required = [col for col in required_columns if col not in df.columns]

if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

display(df.isna().sum().to_frame("missing_values"))

expected_months = pd.date_range(
    start=df["order_month"].min(),
    end=df["order_month"].max(),
    freq="MS"
)

missing_months = expected_months.difference(df["order_month"])

print("Missing months:")
if len(missing_months) == 0:
    print("No missing months inside the observed date range.")
else:
    print(missing_months)

,missing_values
order_month,0
nominal_revenue,0
real_revenue,0
order_count,0
customer_count,0
units_sold,0
units_per_order,0
nominal_avg_order_value,0
real_avg_order_value,0
cpi_index_jan_2021_100,0


Missing months:
No missing months inside the observed date range.


## 4. Nominal vs real revenue over time

This plot frames the main business problem.

Nominal revenue is measured in current prices. Real revenue adjusts nominal revenue using CPI, so it is closer to the underlying business performance after inflation.

Because the two series can tell different stories, we model them separately.

In [26]:
revenue_long = df.melt(
    id_vars="order_month",
    value_vars=["nominal_revenue", "real_revenue"],
    var_name="revenue_type",
    value_name="revenue"
)

fig = px.line(
    revenue_long,
    x="order_month",
    y="revenue",
    color="revenue_type",
    markers=True,
    title="Monthly Nominal vs Real Revenue",
    labels={
        "order_month": "Month",
        "revenue": "Revenue",
        "revenue_type": "Revenue type"
    }
)

fig.update_layout(hovermode="x unified")
fig.show()

## 5. Create time-series modeling features

We will create two types of features.

### Feature 1: trend

`t` is simply the month number from the start of the dataset.

Example:

| Month | t |
|---|---:|
| 2021-01 | 0 |
| 2021-02 | 1 |
| 2021-03 | 2 |

This lets the model learn the broad direction of revenue over time.

### Features 2–3: sine/cosine seasonality

The `sin12` and `cos12` features let the model learn a smooth annual cycle.

This is more compact than creating 11 month dummy variables, which would be too much for such a small dataset.

In [27]:
df["t"] = np.arange(len(df))
df["month"] = df["order_month"].dt.month

df["sin12"] = np.sin(2 * np.pi * df["month"] / 12)
df["cos12"] = np.cos(2 * np.pi * df["month"] / 12)

df[["order_month", "t", "month", "sin12", "cos12"]].head(15)

,order_month,t,month,sin12,cos12
0,2021-01-01,0,1,0.50,0.87
1,2021-02-01,1,2,0.87,0.50
2,2021-03-01,2,3,1.00,0.00
3,2021-04-01,3,4,0.87,-0.50
4,2021-05-01,4,5,0.50,-0.87
5,2021-06-01,5,6,0.00,-1.00
6,2021-07-01,6,7,-0.50,-0.87
7,2021-08-01,7,8,-0.87,-0.50
8,2021-09-01,8,9,-1.00,-0.00
9,2021-10-01,9,10,-0.87,0.50


## 6. Chronological train/test split

Because this is a forecasting problem, we should not use a random train/test split.

A random split can let the model train on future months and test on earlier months, which creates leakage.

Instead:

- Train: 2021-01 to 2022-12
- Test: 2023-01 to 2023-06

The test set is small, but this split preserves time order and gives the model two full years of training history.

In [28]:
train = df[df["order_month"] <= "2022-12-01"].copy()
test = df[df["order_month"] >= "2023-01-01"].copy()

print(f"Train period: {train['order_month'].min().date()} to {train['order_month'].max().date()}")
print(f"Test period:  {test['order_month'].min().date()} to {test['order_month'].max().date()}")
print(f"Train rows: {len(train)}")
print(f"Test rows:  {len(test)}")

Train period: 2021-01-01 to 2022-12-01
Test period:  2023-01-01 to 2023-06-01
Train rows: 24
Test rows:  6


## 7. Train/test split

This plot shows the validation design. The model sees the training period and is evaluated on the next observed months.

The same split is used for nominal and real revenue; the plot below uses real revenue for clarity.

In [29]:
split_plot_df = pd.concat([
    train.assign(split="Train"),
    test.assign(split="Test")
])

fig = px.line(
    split_plot_df,
    x="order_month",
    y="real_revenue",
    color="split",
    markers=True,
    title="Chronological Train/Test Split: Real Revenue",
    labels={
        "order_month": "Month",
        "real_revenue": "Real revenue",
        "split": "Split"
    }
)

fig.update_layout(hovermode="x unified")
fig.show()

## 8. Define candidate models

We compare two regression models.

### Model A — log trend

```text
log(revenue) ~ t
```

This captures only the broad trend.

### Model B — log trend + seasonality

```text
log(revenue) ~ t + sin12 + cos12
```

This captures the broad trend plus a smooth yearly seasonal pattern.

Both models use the log of revenue as the training target. This makes the model behave more like it is learning relative/percentage changes rather than only absolute lira changes.

In [30]:
candidate_models = {
    "log_trend": ["t"],
    "log_trend_seasonal": ["t", "sin12", "cos12"]
}

targets = ["nominal_revenue", "real_revenue"]

## 9. Evaluation functions

The function below trains a Ridge regression model on log revenue, predicts the validation period, converts predictions back to normal revenue scale, and reports common regression metrics.

Metrics used:

- **MAE**: average absolute error in revenue units
- **RMSE**: similar to MAE, but penalizes large errors more heavily
- **MAPE**: average percentage error; useful for comparing nominal vs real revenue
- **R²**: included as a secondary metric, not the main decision rule

For this forecasting problem, model choice should prioritize validation error, especially MAPE, MAE, and RMSE.

In [31]:
def evaluate_log_model(train, test, target_col, feature_cols, model_name, alpha=1.0):
    X_train = train[feature_cols]
    X_test = test[feature_cols]

    y_train_log = np.log(train[target_col])
    y_test = test[target_col]

    model = Ridge(alpha=alpha)
    model.fit(X_train, y_train_log)

    pred_log = model.predict(X_test)
    y_pred = np.exp(pred_log)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mape = mean_absolute_percentage_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    predictions = test[["order_month"]].copy()
    predictions["target"] = target_col
    predictions["model"] = model_name
    predictions["actual"] = y_test.values
    predictions["predicted"] = y_pred
    predictions["residual"] = predictions["actual"] - predictions["predicted"]

    metrics = {
        "target": target_col,
        "model": model_name,
        "features": feature_cols,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape,
        "R2": r2,
        "fitted_model": model
    }

    return metrics, predictions


def evaluate_last_value_baseline(train, test, target_col):
    last_value = train[target_col].iloc[-1]
    y_test = test[target_col]
    y_pred = np.repeat(last_value, len(test))

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mape = mean_absolute_percentage_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    predictions = test[["order_month"]].copy()
    predictions["target"] = target_col
    predictions["model"] = "last_value_baseline"
    predictions["actual"] = y_test.values
    predictions["predicted"] = y_pred
    predictions["residual"] = predictions["actual"] - predictions["predicted"]

    metrics = {
        "target": target_col,
        "model": "last_value_baseline",
        "features": [],
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape,
        "R2": r2,
        "fitted_model": None
    }

    return metrics, predictions

## 10. Train and evaluate all candidates

We evaluate each target separately:

- nominal revenue
- real revenue

For each target, we compare:

- last-value baseline
- log trend
- log trend + sine/cos seasonality

In [32]:
all_metrics = []
all_predictions = []

for target in targets:
    # Baseline
    metrics, preds = evaluate_last_value_baseline(train, test, target)
    all_metrics.append(metrics)
    all_predictions.append(preds)

    # Regression candidates
    for model_name, feature_cols in candidate_models.items():
        metrics, preds = evaluate_log_model(
            train=train,
            test=test,
            target_col=target,
            feature_cols=feature_cols,
            model_name=model_name
        )
        all_metrics.append(metrics)
        all_predictions.append(preds)

metrics_df = pd.DataFrame(all_metrics)
validation_predictions = pd.concat(all_predictions, ignore_index=True)

display_cols = ["target", "model", "MAE", "RMSE", "MAPE", "R2"]
metrics_display = metrics_df[display_cols].copy()
metrics_display["MAPE_percent"] = metrics_display["MAPE"] * 100

metrics_display.sort_values(["target", "MAPE"])

,target,model,MAE,RMSE,MAPE,R2,MAPE_percent
0,nominal_revenue,last_value_baseline,"26,329,037.64","36,356,290.96",0.05,-1.10,5.40
1,nominal_revenue,log_trend,"123,431,506.06","126,012,747.84",0.24,-24.26,24.12
2,nominal_revenue,log_trend_seasonal,"144,844,845.08","148,220,892.47",0.28,-33.95,28.25
5,real_revenue,log_trend_seasonal,"24,522,896.92","25,777,174.31",0.12,-9.85,12.06
4,real_revenue,log_trend,"35,470,103.66","36,414,809.50",0.17,-20.65,17.39
3,real_revenue,last_value_baseline,"39,204,400.61","39,977,809.63",0.19,-25.10,19.20


## 11. Model comparison by validation MAPE

This plot compares the models using validation MAPE.

Lower MAPE is better.

The baseline is important because a model is not automatically useful just because it is more complex. It should beat a simple naive forecast.

In [33]:
plot_metrics = metrics_display.copy()

fig = px.bar(
    plot_metrics,
    x="model",
    y="MAPE_percent",
    color="target",
    barmode="group",
    title="Validation MAPE by Model and Target",
    labels={
        "model": "Model",
        "MAPE_percent": "MAPE (%)",
        "target": "Target"
    }
)

fig.update_layout(xaxis_tickangle=-30)
fig.show()

## 12. Select the best model for each target

The code below selects the model with the lowest validation MAPE separately for each target.

This means nominal and real revenue do not have to use the same model. That is acceptable because they behave differently.

In [34]:
regression_metrics_df = metrics_df[
    metrics_df["model"] != "last_value_baseline"
].copy()

best_models = (
    regression_metrics_df
    .sort_values(["target", "MAPE"])
    .groupby("target", as_index=False)
    .first()
)

best_models[["target", "model", "MAE", "RMSE", "MAPE", "R2"]]

,target,model,MAE,RMSE,MAPE,R2
0,nominal_revenue,log_trend,"123,431,506.06","126,012,747.84",0.24,-24.26
1,real_revenue,log_trend_seasonal,"24,522,896.92","25,777,174.31",0.12,-9.85


## 13. Actual vs predicted on the validation period

This plot shows how the selected model behaved on the held-out 2023 months.

It is useful because metrics alone do not show *when* the model was wrong.

In [35]:
selected_validation_predictions = validation_predictions.merge(
    best_models[["target", "model"]],
    on=["target", "model"],
    how="inner"
)

selected_long = selected_validation_predictions.melt(
    id_vars=["order_month", "target", "model"],
    value_vars=["actual", "predicted"],
    var_name="series",
    value_name="revenue"
)

fig = px.line(
    selected_long,
    x="order_month",
    y="revenue",
    color="series",
    facet_row="target",
    markers=True,
    title="Validation Period: Actual vs Predicted for Selected Models",
    labels={
        "order_month": "Month",
        "revenue": "Revenue",
        "series": "Series",
        "target": "Target"
    }
)

fig.update_layout(hovermode="x unified")
fig.show()

## 14. Residuals for selected models

Residuals are calculated as:

```text
actual - predicted
```

Interpretation:

- Positive residual: the model underpredicted revenue
- Negative residual: the model overpredicted revenue

The goal is not to get perfect residuals. The goal is to see whether errors are systematic.

In [36]:
fig = px.bar(
    selected_validation_predictions,
    x="order_month",
    y="residual",
    color="target",
    facet_row="target",
    title="Validation Residuals for Selected Models",
    labels={
        "order_month": "Month",
        "residual": "Actual - predicted",
        "target": "Target"
    }
)

fig.add_hline(y=0, line_dash="dash")
fig.show()

## 15. Refit selected models on all available data

After validation, we refit the selected model for each target on all observed data.

This is standard: once we have used validation to choose the model, we use all available historical data to train the final forecasting model.

In [37]:
def fit_final_log_model(df, target_col, feature_cols, alpha=1.0):
    X = df[feature_cols]
    y_log = np.log(df[target_col])

    model = Ridge(alpha=alpha)
    model.fit(X, y_log)

    return model


final_models = {}

for _, row in best_models.iterrows():
    target = row["target"]
    model_name = row["model"]

    if model_name == "last_value_baseline":
        final_models[target] = {
            "model_name": model_name,
            "feature_cols": [],
            "model": None,
            "last_value": df[target].iloc[-1]
        }
    else:
        feature_cols = candidate_models[model_name]
        model = fit_final_log_model(df, target, feature_cols)

        final_models[target] = {
            "model_name": model_name,
            "feature_cols": feature_cols,
            "model": model,
            "last_value": None
        }

final_models

{'nominal_revenue': {'model_name': 'log_trend',
  'feature_cols': ['t'],
  'model': Ridge(),
  'last_value': None},
 'real_revenue': {'model_name': 'log_trend_seasonal',
  'feature_cols': ['t', 'sin12', 'cos12'],
  'model': Ridge(),
  'last_value': None}}

## 16. Create the future date frame

The observed data ends in 2023. To forecast 2024, we create future monthly rows from 2023-07 to 2024-12.

The 2023-07 to 2023-12 months are included because the dataset ends at 2023-06, and the model needs a continuous time index before reaching 2024.

In [38]:
future = pd.DataFrame({
    "order_month": pd.date_range(
        start=df["order_month"].max() + pd.offsets.MonthBegin(1),
        end="2024-12-01",
        freq="MS"
    )
})

future["t"] = np.arange(len(df), len(df) + len(future))
future["month"] = future["order_month"].dt.month
future["sin12"] = np.sin(2 * np.pi * future["month"] / 12)
future["cos12"] = np.cos(2 * np.pi * future["month"] / 12)

future.head()

,order_month,t,month,sin12,cos12
0,2023-07-01,30,7,-0.50,-0.87
1,2023-08-01,31,8,-0.87,-0.50
2,2023-09-01,32,9,-1.00,-0.00
3,2023-10-01,33,10,-0.87,0.50
4,2023-11-01,34,11,-0.50,0.87


## 17. Forecast future revenue

The selected model for each target is used to generate forecasts.

If the selected model is the last-value baseline, the forecast simply repeats the last observed value.

In [39]:
forecast = future[["order_month"]].copy()

for target in targets:
    model_info = final_models[target]
    pred_col = f"{target}_forecast"

    if model_info["model_name"] == "last_value_baseline":
        forecast[pred_col] = model_info["last_value"]
    else:
        feature_cols = model_info["feature_cols"]
        model = model_info["model"]
        forecast[pred_col] = np.exp(model.predict(future[feature_cols]))

forecast.head()

,order_month,nominal_revenue_forecast,real_revenue_forecast
0,2023-07-01,"615,012,530.53","219,181,124.31"
1,2023-08-01,"631,869,841.76","221,554,391.70"
2,2023-09-01,"649,189,206.89","220,838,048.92"
3,2023-10-01,"666,983,290.69","216,566,789.78"
4,2023-11-01,"685,265,105.05","209,381,460.54"


## 18. Historical revenue and 2024 forecast

This plot shows the historical actuals and the 2024 forecast.

The gap between 2023-06 and 2024-01 exists because the source data ends at 2023-06, while the final reporting goal is 2024. The model internally also forecasts 2023-07 to 2023-12 to keep the monthly time index continuous.

In [40]:
forecast_2024 = forecast[forecast["order_month"].dt.year == 2024].copy()

history_long = df.melt(
    id_vars="order_month",
    value_vars=["nominal_revenue", "real_revenue"],
    var_name="target",
    value_name="revenue"
)
history_long["series"] = "Historical actual"

forecast_long = forecast_2024.rename(columns={
    "nominal_revenue_forecast": "nominal_revenue",
    "real_revenue_forecast": "real_revenue"
}).melt(
    id_vars="order_month",
    value_vars=["nominal_revenue", "real_revenue"],
    var_name="target",
    value_name="revenue"
)
forecast_long["series"] = "2024 forecast"

history_forecast_long = pd.concat(
    [history_long, forecast_long],
    ignore_index=True
)

fig = px.line(
    history_forecast_long,
    x="order_month",
    y="revenue",
    color="series",
    facet_row="target",
    markers=True,
    title="Historical Revenue and 2024 Forecast",
    labels={
        "order_month": "Month",
        "revenue": "Revenue",
        "series": "Series",
        "target": "Target"
    }
)

fig.update_layout(hovermode="x unified")
fig.show()

## 19. Monthly 2024 forecast table

The table below contains the final monthly 2024 forecasts for nominal and real revenue.

In [41]:
forecast_2024_display = forecast_2024.copy()

forecast_2024_display["order_month"] = forecast_2024_display["order_month"].dt.strftime("%Y-%m")

forecast_2024_display

,order_month,nominal_revenue_forecast,real_revenue_forecast
6,2024-01,"723,345,766.09","192,448,788.75"
7,2024-02,"743,172,459.20","185,960,777.89"
8,2024-03,"763,542,596.10","182,226,360.25"
9,2024-04,"784,471,072.41","181,499,999.55"
10,2024-05,"805,973,192.06","183,363,819.69"
11,2024-06,"828,064,678.43","186,794,719.56"
12,2024-07,"850,761,685.90","190,328,340.10"
13,2024-08,"874,080,811.62","192,389,192.94"
14,2024-09,"898,039,107.66","191,767,148.80"
15,2024-10,"922,654,093.50","188,058,153.95"


## 20. Yearly 2024 forecast totals

The yearly forecast is calculated by summing the monthly 2024 predictions.

In [42]:
yearly_2024_forecast = pd.DataFrame({
    "metric": ["nominal_revenue_forecast", "real_revenue_forecast"],
    "forecast_2024_total": [
        forecast_2024["nominal_revenue_forecast"].sum(),
        forecast_2024["real_revenue_forecast"].sum()
    ]
})

yearly_2024_forecast

,metric,forecast_2024_total
0,nominal_revenue_forecast,"10,115,975,858.39"
1,real_revenue_forecast,"2,230,985,530.71"


## 21. Suggested interpretation

Use the final results cautiously because the dataset is small and ends in 2023-06.

A suitable interpretation would be:

> I modeled nominal and inflation-adjusted real revenue separately because high inflation can make nominal growth misleading. I compared a simple log-trend model, a log-trend model with compact sine/cosine seasonality, and a last-value baseline. The models were evaluated using a chronological split, training on 2021–2022 and testing on the first half of 2023. Final model selection was based primarily on validation MAPE, with MAE and RMSE used as supporting metrics. The selected model was then refit on all available historical data and used to forecast monthly 2024 revenue.

Important limitations:

- The dataset has only about 30 monthly observations.
- The validation set has only 6 months.
- The source data ends at 2023-06, so 2024 is a relatively long forecast horizon.
- Nominal revenue depends heavily on future inflation and pricing conditions.
- The real revenue forecast is usually more meaningful for business performance.

# 22. Ingest results into BigQuery


In [44]:
# ---------------------------------------------------------------------
# Create Power BI forecast mart
# ---------------------------------------------------------------------
# Assumes these already exist from earlier notebook cells:
# - df
# - forecast
# - PROJECT_ID
# - DATASET
# - LOCATION
# - final_models
#
# df contains actual historical monthly revenue through 2023-06.
# forecast contains predicted months from 2023-07 to 2024-12.

FORECAST_TABLE_NAME = "mart_monthly_revenue_actual_forecast"
DESTINATION_TABLE = f"{DATASET}.{FORECAST_TABLE_NAME}"

# -----------------------------
# 1. Historical actual rows
# -----------------------------
historical_mart = df[[
    "order_month",
    "nominal_revenue",
    "real_revenue"
]].copy()

historical_mart["is_forecast"] = False
historical_mart["data_status"] = "Actual"

# -----------------------------
# 2. Forecast rows
# -----------------------------
forecast_mart = forecast.rename(columns={
    "nominal_revenue_forecast": "nominal_revenue",
    "real_revenue_forecast": "real_revenue"
})[[
    "order_month",
    "nominal_revenue",
    "real_revenue"
]].copy()

forecast_mart["is_forecast"] = True
forecast_mart["data_status"] = "Forecast"

# -----------------------------
# 3. Combine actual + forecast
# -----------------------------
revenue_powerbi_mart = pd.concat(
    [historical_mart, forecast_mart],
    ignore_index=True
)

revenue_powerbi_mart["order_month"] = pd.to_datetime(
    revenue_powerbi_mart["order_month"]
)

revenue_powerbi_mart = revenue_powerbi_mart.sort_values(
    "order_month"
).reset_index(drop=True)

revenue_powerbi_mart["order_year"] = revenue_powerbi_mart["order_month"].dt.year
revenue_powerbi_mart["order_month_num"] = revenue_powerbi_mart["order_month"].dt.month
revenue_powerbi_mart["order_month_label"] = revenue_powerbi_mart["order_month"].dt.strftime("%Y-%m")

# -----------------------------
# 4. Add Power BI-friendly split columns
# -----------------------------
# These columns support two different visual needs:
#
# 1. *_actual and *_forecast:
#    Use these for yearly stacked bars and totals.
#    Forecast values only exist in true forecast months.
#
# 2. *_forecast_line:
#    Use these only for monthly line charts.
#    They include the last actual point as an anchor, so the forecast
#    line visually connects to the historical line.

# Standard actual/forecast split columns
revenue_powerbi_mart["nominal_revenue_actual"] = np.where(
    revenue_powerbi_mart["is_forecast"] == False,
    revenue_powerbi_mart["nominal_revenue"],
    np.nan
)

revenue_powerbi_mart["nominal_revenue_forecast"] = np.where(
    revenue_powerbi_mart["is_forecast"] == True,
    revenue_powerbi_mart["nominal_revenue"],
    np.nan
)

revenue_powerbi_mart["real_revenue_actual"] = np.where(
    revenue_powerbi_mart["is_forecast"] == False,
    revenue_powerbi_mart["real_revenue"],
    np.nan
)

revenue_powerbi_mart["real_revenue_forecast"] = np.where(
    revenue_powerbi_mart["is_forecast"] == True,
    revenue_powerbi_mart["real_revenue"],
    np.nan
)

# Find the last actual month
last_actual_month = revenue_powerbi_mart.loc[
    revenue_powerbi_mart["is_forecast"] == False,
    "order_month"
].max()

print(f"Last actual month: {last_actual_month}")

# Forecast-line anchor columns
# These should be used in line charts only, not yearly totals.
revenue_powerbi_mart["nominal_revenue_forecast_line"] = np.where(
    (revenue_powerbi_mart["is_forecast"] == True)
    | (revenue_powerbi_mart["order_month"] == last_actual_month),
    revenue_powerbi_mart["nominal_revenue"],
    np.nan
)

revenue_powerbi_mart["real_revenue_forecast_line"] = np.where(
    (revenue_powerbi_mart["is_forecast"] == True)
    | (revenue_powerbi_mart["order_month"] == last_actual_month),
    revenue_powerbi_mart["real_revenue"],
    np.nan
)

# -----------------------------
# 5. Add metadata
# -----------------------------
revenue_powerbi_mart["nominal_model_name"] = final_models["nominal_revenue"]["model_name"]
revenue_powerbi_mart["real_model_name"] = final_models["real_revenue"]["model_name"]

revenue_powerbi_mart["forecast_start_month"] = pd.Timestamp("2023-07-01")
revenue_powerbi_mart["forecast_version"] = "v1"
revenue_powerbi_mart["generated_at_utc"] = pd.Timestamp.utcnow()

# -----------------------------
# 6. Clean final dtypes
# -----------------------------
float_cols = [
    "nominal_revenue",
    "real_revenue",
    "nominal_revenue_actual",
    "nominal_revenue_forecast",
    "real_revenue_actual",
    "real_revenue_forecast",
    "nominal_revenue_forecast_line",
    "real_revenue_forecast_line"
]

for col in float_cols:
    revenue_powerbi_mart[col] = revenue_powerbi_mart[col].astype(float)

revenue_powerbi_mart["order_year"] = revenue_powerbi_mart["order_year"].astype("int64")
revenue_powerbi_mart["order_month_num"] = revenue_powerbi_mart["order_month_num"].astype("int64")
revenue_powerbi_mart["is_forecast"] = revenue_powerbi_mart["is_forecast"].astype(bool)

# BigQuery DATE fields are safer if passed as Python date objects with explicit schema.
revenue_powerbi_mart["order_month"] = revenue_powerbi_mart["order_month"].dt.date
revenue_powerbi_mart["forecast_start_month"] = pd.to_datetime(
    revenue_powerbi_mart["forecast_start_month"]
).dt.date

# Preview before upload
display(revenue_powerbi_mart.head())
display(revenue_powerbi_mart.tail())

print(f"Rows to upload: {len(revenue_powerbi_mart)}")
print(f"Date range: {revenue_powerbi_mart['order_month'].min()} to {revenue_powerbi_mart['order_month'].max()}")
print(f"Destination: {PROJECT_ID}.{DESTINATION_TABLE}")

# -----------------------------
# 7. Upload to BigQuery
# -----------------------------
table_schema = [
    {"name": "order_month", "type": "DATE"},
    {"name": "order_year", "type": "INTEGER"},
    {"name": "order_month_num", "type": "INTEGER"},
    {"name": "order_month_label", "type": "STRING"},

    {"name": "nominal_revenue", "type": "FLOAT"},
    {"name": "real_revenue", "type": "FLOAT"},

    {"name": "nominal_revenue_actual", "type": "FLOAT"},
    {"name": "nominal_revenue_forecast", "type": "FLOAT"},
    {"name": "nominal_revenue_forecast_line", "type": "FLOAT"},
    {"name": "real_revenue_actual", "type": "FLOAT"},
    {"name": "real_revenue_forecast", "type": "FLOAT"},
    {"name": "real_revenue_forecast_line", "type": "FLOAT"},

    {"name": "is_forecast", "type": "BOOLEAN"},
    {"name": "data_status", "type": "STRING"},

    {"name": "nominal_model_name", "type": "STRING"},
    {"name": "real_model_name", "type": "STRING"},
    {"name": "forecast_start_month", "type": "DATE"},
    {"name": "forecast_version", "type": "STRING"},
    {"name": "generated_at_utc", "type": "TIMESTAMP"},
]

pandas_gbq.to_gbq(
    dataframe=revenue_powerbi_mart,
    destination_table=DESTINATION_TABLE,
    project_id=PROJECT_ID,
    if_exists="replace",
    table_schema=table_schema,
    location=LOCATION
)

print(f"Uploaded table: {PROJECT_ID}.{DESTINATION_TABLE}")

Last actual month: 2023-06-01 00:00:00


/var/folders/28/8k26_pcd2bz699b6mx6vz2yc0000gn/T/ipykernel_2072/3017309461.py:136: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  revenue_powerbi_mart["generated_at_utc"] = pd.Timestamp.utcnow()


,order_month,nominal_revenue,real_revenue,is_forecast,data_status,order_year,order_month_num,order_month_label,nominal_revenue_actual,nominal_revenue_forecast,real_revenue_actual,real_revenue_forecast,nominal_revenue_forecast_line,real_revenue_forecast_line,nominal_model_name,real_model_name,forecast_start_month,forecast_version,generated_at_utc
0,2021-01-01,"262,374,739.35","262,374,739.35",False,Actual,2021,1,2021-01,"262,374,739.35",NaN,"262,374,739.35",NaN,NaN,NaN,log_trend,log_trend_seasonal,2023-07-01,v1,2026-06-04 09:22:06.796329+00:00
1,2021-02-01,"245,348,178.10","243,140,821.27",False,Actual,2021,2,2021-02,"245,348,178.10",NaN,"243,140,821.27",NaN,NaN,NaN,log_trend,log_trend_seasonal,2023-07-01,v1,2026-06-04 09:22:06.796329+00:00
2,2021-03-01,"280,547,713.26","275,065,691.15",False,Actual,2021,3,2021-03,"280,547,713.26",NaN,"275,065,691.15",NaN,NaN,NaN,log_trend,log_trend_seasonal,2023-07-01,v1,2026-06-04 09:22:06.796329+00:00
3,2021-04-01,"281,199,642.31","271,152,269.98",False,Actual,2021,4,2021-04,"281,199,642.31",NaN,"271,152,269.98",NaN,NaN,NaN,log_trend,log_trend_seasonal,2023-07-01,v1,2026-06-04 09:22:06.796329+00:00
4,2021-05-01,"300,710,698.34","287,412,348.08",False,Actual,2021,5,2021-05,"300,710,698.34",NaN,"287,412,348.08",NaN,NaN,NaN,log_trend,log_trend_seasonal,2023-07-01,v1,2026-06-04 09:22:06.796329+00:00


,order_month,nominal_revenue,real_revenue,is_forecast,data_status,order_year,order_month_num,order_month_label,nominal_revenue_actual,nominal_revenue_forecast,real_revenue_actual,real_revenue_forecast,nominal_revenue_forecast_line,real_revenue_forecast_line,nominal_model_name,real_model_name,forecast_start_month,forecast_version,generated_at_utc
43,2024-08-01,"874,080,811.62","192,389,192.94",True,Forecast,2024,8,2024-08,NaN,"874,080,811.62",NaN,"192,389,192.94","874,080,811.62","192,389,192.94",log_trend,log_trend_seasonal,2023-07-01,v1,2026-06-04 09:22:06.796329+00:00
44,2024-09-01,"898,039,107.66","191,767,148.80",True,Forecast,2024,9,2024-09,NaN,"898,039,107.66",NaN,"191,767,148.80","898,039,107.66","191,767,148.80",log_trend,log_trend_seasonal,2023-07-01,v1,2026-06-04 09:22:06.796329+00:00
45,2024-10-01,"922,654,093.50","188,058,153.95",True,Forecast,2024,10,2024-10,NaN,"922,654,093.50",NaN,"188,058,153.95","922,654,093.50","188,058,153.95",log_trend,log_trend_seasonal,2023-07-01,v1,2026-06-04 09:22:06.796329+00:00
46,2024-11-01,"947,943,768.81","181,818,694.27",True,Forecast,2024,11,2024-11,NaN,"947,943,768.81",NaN,"181,818,694.27","947,943,768.81","181,818,694.27",log_trend,log_trend_seasonal,2023-07-01,v1,2026-06-04 09:22:06.796329+00:00
47,2024-12-01,"973,926,626.62","174,329,534.96",True,Forecast,2024,12,2024-12,NaN,"973,926,626.62",NaN,"174,329,534.96","973,926,626.62","174,329,534.96",log_trend,log_trend_seasonal,2023-07-01,v1,2026-06-04 09:22:06.796329+00:00


Rows to upload: 48
Date range: 2021-01-01 to 2024-12-01
Destination: superstore-analysis-496710.dbt_doruk.mart_monthly_revenue_actual_forecast
Uploaded table: superstore-analysis-496710.dbt_doruk.mart_monthly_revenue_actual_forecast


In [45]:
# ---------------------------------------------------------------------
# Create yearly display mart for Power BI bar chart
# ---------------------------------------------------------------------
# Purpose:
# Simple annual chart with nominal and real revenue totals.
#
# 2021-2022: actual totals
# 2023: actual Jan-Jun + forecast Jul-Dec
# 2024: forecast total
#
# This table is intended for presentation-friendly yearly bar charts,
# not detailed model evaluation.

YEARLY_FORECAST_TABLE_NAME = "mart_yearly_revenue_actual_forecast_display"
YEARLY_DESTINATION_TABLE = f"{DATASET}.{YEARLY_FORECAST_TABLE_NAME}"

# Use the already-created monthly actual + forecast mart
yearly_display_mart = (
    revenue_powerbi_mart
    .groupby("order_year", as_index=False)
    .agg(
        nominal_revenue=("nominal_revenue", "sum"),
        real_revenue=("real_revenue", "sum"),
        actual_months=("is_forecast", lambda s: int((s == False).sum())),
        forecast_months=("is_forecast", lambda s: int((s == True).sum()))
    )
)

# Assign yearly status
def assign_year_status(row):
    if row["actual_months"] > 0 and row["forecast_months"] == 0:
        return "Actual"
    elif row["actual_months"] > 0 and row["forecast_months"] > 0:
        return "Actual + Forecast"
    elif row["actual_months"] == 0 and row["forecast_months"] > 0:
        return "Forecast"
    else:
        return "Unknown"

yearly_display_mart["year_status"] = yearly_display_mart.apply(
    assign_year_status,
    axis=1
)

# Presentation-friendly labels
yearly_display_mart["year_label"] = yearly_display_mart.apply(
    lambda row: (
        f"{row['order_year']} (Actual + Forecast)"
        if row["year_status"] == "Actual + Forecast"
        else f"{row['order_year']} (Forecast)"
        if row["year_status"] == "Forecast"
        else str(row["order_year"])
    ),
    axis=1
)

# Useful for sorting the label correctly in Power BI
yearly_display_mart["year_sort"] = yearly_display_mart["order_year"].astype(int)

# Optional metadata
yearly_display_mart["forecast_version"] = "v1"
yearly_display_mart["generated_at_utc"] = pd.Timestamp.utcnow()

# Clean dtypes
yearly_display_mart["order_year"] = yearly_display_mart["order_year"].astype("int64")
yearly_display_mart["year_sort"] = yearly_display_mart["year_sort"].astype("int64")
yearly_display_mart["actual_months"] = yearly_display_mart["actual_months"].astype("int64")
yearly_display_mart["forecast_months"] = yearly_display_mart["forecast_months"].astype("int64")
yearly_display_mart["nominal_revenue"] = yearly_display_mart["nominal_revenue"].astype(float)
yearly_display_mart["real_revenue"] = yearly_display_mart["real_revenue"].astype(float)

display(yearly_display_mart)

print(f"Rows to upload: {len(yearly_display_mart)}")
print(f"Destination: {PROJECT_ID}.{YEARLY_DESTINATION_TABLE}")

# ---------------------------------------------------------------------
# Upload to BigQuery
# ---------------------------------------------------------------------
yearly_table_schema = [
    {"name": "order_year", "type": "INTEGER"},
    {"name": "year_label", "type": "STRING"},
    {"name": "year_sort", "type": "INTEGER"},
    {"name": "year_status", "type": "STRING"},

    {"name": "nominal_revenue", "type": "FLOAT"},
    {"name": "real_revenue", "type": "FLOAT"},

    {"name": "actual_months", "type": "INTEGER"},
    {"name": "forecast_months", "type": "INTEGER"},

    {"name": "forecast_version", "type": "STRING"},
    {"name": "generated_at_utc", "type": "TIMESTAMP"},
]

pandas_gbq.to_gbq(
    dataframe=yearly_display_mart,
    destination_table=YEARLY_DESTINATION_TABLE,
    project_id=PROJECT_ID,
    if_exists="replace",
    table_schema=yearly_table_schema,
    location=LOCATION
)

print(f"Uploaded table: {PROJECT_ID}.{YEARLY_DESTINATION_TABLE}")

/var/folders/28/8k26_pcd2bz699b6mx6vz2yc0000gn/T/ipykernel_2072/1342346881.py:62: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  yearly_display_mart["generated_at_utc"] = pd.Timestamp.utcnow()


,order_year,nominal_revenue,real_revenue,actual_months,forecast_months,year_status,year_label,year_sort,forecast_version,generated_at_utc
0,2021,"3,708,691,848.04","3,383,648,140.18",12,0,Actual,2021,2021,v1,2026-06-04 09:54:26.488657+00:00
1,2022,"5,701,541,839.01","3,040,973,257.84",12,0,Actual,2022,2022,v1,2026-06-04 09:54:26.488657+00:00
2,2023,"7,029,442,102.62","2,524,584,418.40",6,6,Actual + Forecast,2023 (Actual + Forecast),2023,v1,2026-06-04 09:54:26.488657+00:00
3,2024,"10,115,975,858.39","2,230,985,530.71",0,12,Forecast,2024 (Forecast),2024,v1,2026-06-04 09:54:26.488657+00:00


Rows to upload: 4
Destination: superstore-analysis-496710.dbt_doruk.mart_yearly_revenue_actual_forecast_display
Uploaded table: superstore-analysis-496710.dbt_doruk.mart_yearly_revenue_actual_forecast_display
